### Importamos las librerias necesarias y importamos la demanda

In [2]:
!pip install pulp

In [3]:
import pandas as pd
import numpy as np
import pulp
import plotly.express as px

# 1. Cargar las bases de datos de la empresa
df_ventas = pd.read_csv('historial_ventas_lavandina.csv')
df_equipos = pd.read_csv('maestro_equipos.csv')
df_recetas = pd.read_csv('maestro_recetas.csv')
df_mantenimiento = pd.read_csv('mantenimiento.csv')

print("Datos cargados exitosamente.")
display(df_ventas.head(3))
display(df_equipos.head(2))

Datos cargados exitosamente.


,Fecha,SKU,Producto,Litros_Vendidos
0,2023-10-01,LAV-FLO-02,Lavandina Floral,6971
1,2023-10-02,LAV-BLA-03,Lavandina Ropa Blanca,5467
2,2023-10-03,LAV-CLA-01,Lavandina Clásica,11365


,ID_Tanque,Tipo,Capacidad_L,Tiempo_CIP_hs,Costo_CIP_ARS,Potencia_Agitador_kW
0,TK-100,Reactor Batch,10000,2.0,18000,15.0
1,TK-200,Reactor Batch,5000,1.5,12000,8.0


### Demanda Comercial y Costos

In [4]:
# --- A. CÁLCULO DE DEMANDA PROYECTADA ---
# Agrupamos el historial de ventas para obtener el volumen total mensual requerido por SKU
df_demanda = df_ventas.groupby(['SKU', 'Producto'])['Litros_Vendidos'].sum().reset_index()
df_demanda.rename(columns={'Litros_Vendidos': 'Demanda_Proyectada_L'}, inplace=True)

# Supongamos unos precios de venta de lista (esto podrías tenerlo en otro CSV a futuro)
precios = {'LAV-CLA-01': 120.0, 'LAV-FLO-02': 250.0, 'LAV-BLA-03': 210.0}
df_demanda['Precio_Venta_ARS'] = df_demanda['SKU'].map(precios)

# --- B. CÁLCULO DE COSTOS DESDE RECETAS ---
# Multiplicamos la fracción del insumo por su costo unitario
df_recetas['Costo_Aporte'] = df_recetas['Fraccion_Receta'] * df_recetas['Costo_Unitario_ARS']
df_costos = df_recetas.groupby('SKU_Producto')['Costo_Aporte'].sum().reset_index()
df_costos.rename(columns={'SKU_Producto': 'SKU', 'Costo_Aporte': 'Costo_Prod_L'}, inplace=True)

# --- C. CONSOLIDACIÓN COMERCIAL (MARGEN S&OP) ---
df_sop = pd.merge(df_demanda, df_costos, on='SKU')
df_sop['Margen_Unitario'] = df_sop['Precio_Venta_ARS'] - df_sop['Costo_Prod_L']

print("Tabla Táctica S&OP (Demanda y Márgenes):")
display(df_sop)

Tabla Táctica S&OP (Demanda y Márgenes):


,SKU,Producto,Demanda_Proyectada_L,Precio_Venta_ARS,Costo_Prod_L,Margen_Unitario
0,LAV-BLA-03,Lavandina Ropa Blanca,3448723,210.0,127.275,82.725
1,LAV-CLA-01,Lavandina Clásica,11268507,120.0,71.750,48.250
2,LAV-FLO-02,Lavandina Floral,5378340,250.0,143.350,106.650


### (Capacidad Física y Mantenimiento)

In [5]:
# --- CÁLCULO DE CAPACIDAD FÍSICA MENSUAL ---
# Asumimos un mes de 30 días, trabajando 24 hs (720 horas totales por tanque)
HORAS_MES = 720

# Restamos las horas de mantenimiento programado por tanque
mant_horas = df_mantenimiento.groupby('Tanque')['Horas'].sum().to_dict()

capacidad_real_L = 0
for index, row in df_equipos.iterrows():
    tanque = row['ID_Tanque']
    capacidad_lote = row['Capacidad_L']
    tiempo_cip = row['Tiempo_CIP_hs']
    
    # Tiempo disponible real del tanque en el mes
    horas_disponibles = HORAS_MES - mant_horas.get(tanque, 0)
    
    # Estimación simplificada: Cuántos lotes entran en las horas disponibles 
    # (Asumiendo un tiempo promedio de proceso por lote de 5hs + CIP)
    tiempo_ciclo_promedio = 5 + tiempo_cip 
    lotes_posibles = horas_disponibles // tiempo_ciclo_promedio
    
    capacidad_real_L += lotes_posibles * capacidad_lote

print(f"Capacidad Total Ajustada (contemplando mantenimiento): {capacidad_real_L:,.0f} Litros")

Capacidad Total Ajustada (contemplando mantenimiento): 1,795,000 Litros


### Optimización MILP (Maximizar Ganancia)

In [6]:
# Inicializar modelo de MAXIMIZACIÓN
modelo = pulp.LpProblem("SOP_Tactico", pulp.LpMaximize)

skus = df_sop['SKU'].tolist()
produccion = pulp.LpVariable.dicts("Prod_L", skus, lowBound=0, cat='Continuous')

# 1. FUNCIÓN OBJETIVO: Maximizar el margen bruto total
modelo += pulp.lpSum(
    produccion[sku] * df_sop.loc[df_sop['SKU'] == sku, 'Margen_Unitario'].values[0] 
    for sku in skus
), "Max_Margen_Bruto"

# 2. RESTRICCIÓN DE MERCADO: No fabricar más de lo que marca la proyección de ventas
for sku in skus:
    demanda = df_sop.loc[df_sop['SKU'] == sku, 'Demanda_Proyectada_L'].values[0]
    modelo += produccion[sku] <= demanda, f"Lim_Demanda_{sku}"

# 3. RESTRICCIÓN DE PLANTA: El total producido no puede superar la capacidad física
modelo += pulp.lpSum(produccion[sku] for sku in skus) <= capacidad_real_L, "Lim_Capacidad_Planta"

modelo.solve()
print(f"Estado de Resolución S&OP: {pulp.LpStatus[modelo.status]}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/eliseo/anaconda3/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/805ac2b476804cfcb6cfd7f3d79a6c55-pulp.mps -max -timeMode elapsed -solve -printingOptions all -solution /tmp/805ac2b476804cfcb6cfd7f3d79a6c55-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 19 RHS
At line 24 BOUNDS
At line 25 ENDATA
Problem MODEL has 4 rows, 3 columns and 6 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 0 (-4) rows, 0 (-3) columns and 0 (-6) elements
Empty problem - 0 rows, 0 columns and 0 elements
Optimal - objective value 1.9143675e+08
After Postsolve, objective 1.9143675e+08, infeasibilities - dual 0 (0), primal 0 (0)
Optimal objective 191436750 - 0 iterations time 0.002, Presolve 0.00
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   

### Resultados para la Gerencia

In [7]:
# =============================================================================
# CELDA 5: RESULTADOS Y EXPORTACIÓN PARA STREAMLIT
# =============================================================================

# 1. Extraemos los resultados de las variables optimizadas por PuLP
resultados = []
for sku in skus:
    vol = produccion[sku].varValue  # Usando la sintaxis de PuLP
    demanda = df_sop.loc[df_sop['SKU'] == sku, 'Demanda_Proyectada_L'].values[0]
    
    resultados.append({
        'SKU': sku,
        'Demanda_Proyectada (L)': demanda,
        'Produccion_Asignada (L)': vol,
        'Nivel_Servicio (%)': round((vol / demanda) * 100, 1) if demanda > 0 else 0
    })

# 2. Creamos el DataFrame final
df_resultados = pd.DataFrame(resultados)

# 3. Lo mostramos en pantalla dentro del Jupyter para validar
print("--- PLAN DE PRODUCCIÓN S&OP ---")
display(df_resultados)

ganancia = pulp.value(modelo.objective)
print(f"\nGanancia Bruta Maximizada: ${ganancia:,.2f} ARS")

# 4. ¡LA PARTE CLAVE! Exportamos a CSV para levantar con Streamlit
df_resultados.to_csv('resultados_sop.csv', index=False)
print("\n¡Archivo 'resultados_sop.csv' guardado con éxito! Listo para el Dashboard.")

--- PLAN DE PRODUCCIÓN S&OP ---


,SKU,Demanda_Proyectada (L),Produccion_Asignada (L),Nivel_Servicio (%)
0,LAV-BLA-03,3448723,0.0,0.0
1,LAV-CLA-01,11268507,0.0,0.0
2,LAV-FLO-02,5378340,1795000.0,33.4



Ganancia Bruta Maximizada: $191,436,750.00 ARS

¡Archivo 'resultados_sop.csv' guardado con éxito! Listo para el Dashboard.
